In [1]:
import pandas as pd

try:
    # Ganti path sesuai lokasi file kamu
    df_meal = pd.read_csv('../../datas/dataset_meal.csv') 
    
    print("--- DAFTAR KOLOM DATASET MEAL ---")
    print(df_meal.columns.tolist())
    
    print("\n--- CONTOH 3 BARIS DATA ---")
    print(df_meal.head(3).T) # .T biar tampilannya ke bawah, enak dibaca
    
except FileNotFoundError:
    print("File dataset_meal.csv tidak ditemukan. Cek path-nya ya!")

--- DAFTAR KOLOM DATASET MEAL ---
['Food Items', 'Energy kcal', 'Carbs', 'Protein(g)', 'Fat(g)', 'Freesugar(g)', 'Fibre(g)', 'Cholestrol(mg)', 'Calcium(mg)']

--- CONTOH 3 BARIS DATA ---
                         0        1       2
Food Items      Butternaan  Cupcake  Donuts
Energy kcal          300.0    200.0   250.0
Carbs                 50.0     30.0    30.0
Protein(g)             7.0      2.0     3.0
Fat(g)                10.0      8.0    12.0
Freesugar(g)           2.0     20.0    10.0
Fibre(g)               2.0      0.5     1.0
Cholestrol(mg)        15.0     20.0    20.0
Calcium(mg)           50.0     20.0    20.0


In [2]:
import pandas as pd
import numpy as np
import pickle
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# ==========================================
# 1. LOAD DATA & CLEANING
# ==========================================
print("--- LOAD DATASET MEAL ---")
try:
    df_meal = pd.read_csv('../../datas/dataset_meal.csv')
    # Hapus spasi di nama kolom agar mudah dipanggil
    df_meal.columns = df_meal.columns.str.strip()
    print(f"Data Loaded: {df_meal.shape} baris.")
except:
    print("File dataset_meal.csv tidak ditemukan.")
    exit()

# Bersihkan Data: Hapus duplikat makanan jika ada
df_meal = df_meal.drop_duplicates(subset=['Food Items']).reset_index(drop=True)

# Ganti nama kolom biar coding-nya enak (opsional tapi sangat disarankan)
# Sesuaikan dengan output kolom kamu tadi
df_meal = df_meal.rename(columns={
    'Energy kcal': 'Energy',
    'Protein(g)': 'Protein',
    'Fat(g)': 'Fat',
    'Carbs': 'Carbs', # Kadang namanya Carbs saja
    'Fibre(g)': 'Fibre'
})

# ==========================================
# 2. FEATURE SELECTION & SCALING
# ==========================================
# Fitur utama untuk pencarian (Macro Nutrients)
# Kita cari makanan berdasarkan: Kalori, Protein, Lemak, Karbo
features_nutrisi = ['Energy', 'Protein', 'Fat', 'Carbs']

# Cek apakah ada nilai kosong/NaN
if df_meal[features_nutrisi].isnull().sum().sum() > 0:
    print("Ada nilai kosong, mengisi dengan 0...")
    df_meal[features_nutrisi] = df_meal[features_nutrisi].fillna(0)

# SCALING (Sangat Penting!)
# Kalori (misal 500) jauh lebih besar dari Protein (misal 20).
# Tanpa scaling, KNN hanya akan peduli pada Kalori.
scaler_meal = MinMaxScaler()
features_scaled = scaler_meal.fit_transform(df_meal[features_nutrisi])

# Simpan dalam DataFrame biar rapi
df_features = pd.DataFrame(features_scaled, columns=features_nutrisi)

# ==========================================
# 3. SPLIT & EVALUASI MODEL
# ==========================================
print("\n--- MULAI EVALUASI MODEL ---")

# Kita split: 
# Train = Database Makanan di "Kulkas"
# Test  = Skenario permintaan user (misal: "Saya mau makanan dengan gizi X")
X_train, X_test, idx_train, idx_test = train_test_split(
    df_features, df_meal.index, test_size=0.1, random_state=42
)

# Latih KNN pada Database Makanan
# n_neighbors=3 artinya: "Berikan 3 opsi makanan terdekat"
knn_eval = NearestNeighbors(n_neighbors=3, metric='euclidean')
knn_eval.fit(X_train)

# Prediksi: Cari makanan untuk data Test
distances, indices = knn_eval.kneighbors(X_test)

# --- HITUNG ERROR / PENYIMPANGAN ---
# Kita ingin tahu: Seberapa melenceng gizi makanan yang disarankan
# dibanding gizi yang diminta?
total_diff_cal = 0
total_diff_prot = 0
count = len(X_test)

results = []

for i in range(count):
    # Ini adalah "Makanan Target" (yang diminta)
    idx_target = idx_test[i]
    target_data = df_meal.iloc[idx_target]
    
    # Ini adalah "Makanan Rekomendasi" (Opsi 1)
    idx_rec = idx_train[indices[i][0]]
    rec_data = df_meal.iloc[idx_rec]
    
    diff_cal = abs(target_data['Energy'] - rec_data['Energy'])
    diff_prot = abs(target_data['Protein'] - rec_data['Protein'])
    
    total_diff_cal += diff_cal
    total_diff_prot += diff_prot
    
    # Simpan contoh untuk laporan
    if i < 5: # Ambil 5 contoh teratas
        results.append({
            'Target_Food': target_data['Food Items'],
            'Rec_Food': rec_data['Food Items'],
            'Target_Cal': target_data['Energy'],
            'Rec_Cal': rec_data['Energy'],
            'Diff_Cal': diff_cal
        })

avg_diff_cal = total_diff_cal / count
avg_diff_prot = total_diff_prot / count

print(f"Total Data Test (Skenario): {count}")
print(f"Rata-rata Penyimpangan Kalori : {avg_diff_cal:.2f} kcal")
print(f"Rata-rata Penyimpangan Protein: {avg_diff_prot:.2f} g")

print("\n--- CONTOH HASIL PENCARIAN (Top 5) ---")
print(pd.DataFrame(results).to_string(index=False))

# ==========================================
# 4. TRAINING FINAL & SIMPAN MODEL
# ==========================================
print("\n--- MENYIMPAN MODEL FINAL ---")

# Latih dengan 100% Data
knn_final = NearestNeighbors(n_neighbors=5, metric='euclidean')
knn_final.fit(df_features) # Pakai data yang sudah discale

data_meal_model = {
    'knn_model': knn_final,
    'scaler': scaler_meal,
    'meal_db': df_meal,       # Database lengkap
    'features': features_nutrisi
}

with open('../../models/model_meal.pickle', 'wb') as f:
    pickle.dump(data_meal_model, f)

print("Sukses! 'model_meal.pickle' telah disimpan.")

--- LOAD DATASET MEAL ---
Data Loaded: (1028, 9) baris.

--- MULAI EVALUASI MODEL ---
Total Data Test (Skenario): 103
Rata-rata Penyimpangan Kalori : 9.64 kcal
Rata-rata Penyimpangan Protein: 0.33 g

--- CONTOH HASIL PENCARIAN (Top 5) ---
                                                     Target_Food                                                                               Rec_Food  Target_Cal  Rec_Cal  Diff_Cal
                                                Soyabean muthias                                                                            Masala vada      839.33   826.02     13.31
Rasam with tamarind (Puli rasam/ Chintapandu rasam/ Charu/Saaru) Rasam with lemon (Nimmakaya rasam/Nimmakaya charu/Elumichai rasam/Nimbe hannina saaru)       26.74    24.41      2.33
                                                    Cheese balls                                                                  Vegeterian scotch egg      681.28   681.67      0.39
                             